In [11]:
import pandas as pd

In [12]:
FOLDER = "ParquetNIBRS"

In [13]:
df_offense = pd.read_parquet(FOLDER + '/NIBRS_OFFENSE.parquet')
df_offense = df_offense[['OFFENSE_ID','INCIDENT_ID','OFFENSE_CODE']]
df_offense = df_offense[df_offense['OFFENSE_CODE'].notna()]
df_offense.head(5)

,OFFENSE_ID,INCIDENT_ID,OFFENSE_CODE
97608,172305650,143439540,13B
97609,172305651,143439541,13B
97610,172305652,143439542,11A
97611,172305654,143439544,13B
97612,172305655,143439545,23F


In [14]:
df_incident = pd.read_parquet(FOLDER + '/NIBRS_INCIDENT.parquet')
df_incident = df_incident[['AGENCY_ID','INCIDENT_ID','INCIDENT_DATE','INCIDENT_HOUR']]

In [15]:
incident_date_list = list(df_incident.INCIDENT_DATE.unique())
df_incident_date = pd.DataFrame({'DATE':incident_date_list})
df_incident_date['YEAR_LEN'] = df_incident_date.DATE.apply(lambda x: len(x.split("-")[0]))
df_incident_date_issue = df_incident_date.copy().loc[df_incident_date.YEAR_LEN < 4]

In [16]:
df_incident_date_issue["DATE_CORRECT"] = pd.to_datetime(
    df_incident_date_issue["DATE"],
    format="%d-%b-%y"
)

df_incident_date_issue["DATE_CORRECT"] = df_incident_date_issue["DATE_CORRECT"].dt.strftime("%Y-%m-%d")
incident_date_dict = dict(
    zip(
        df_incident_date_issue["DATE"],
        df_incident_date_issue["DATE_CORRECT"]
    )
)

In [17]:
df_incident["INCIDENT_DATE"] = (
    df_incident["INCIDENT_DATE"]
    .map(incident_date_dict)
    .fillna(df_incident["INCIDENT_DATE"])
)

df_incident['INCIDENT_DATE'] = df_incident.INCIDENT_DATE.apply(lambda x: x.split(" ")[0])

AGENCY_ID, INCIDENT_ID, NIBRS_MONTH_ID, INCIDENT_DATE, INCIDENT_HOUR, 

In [18]:
df_incident["INCIDENT_DATETIME"] = (
    pd.to_datetime(df_incident["INCIDENT_DATE"]) +
    pd.to_timedelta(df_incident["INCIDENT_HOUR"], unit="h")
)

In [19]:
df_incident.head(5)

,AGENCY_ID,INCIDENT_ID,INCIDENT_DATE,INCIDENT_HOUR,INCIDENT_DATETIME
0,5659,78274377,2015-01-02,0.0,2015-01-02 00:00:00
1,5659,78274378,2015-01-02,21.0,2015-01-02 21:00:00
2,5659,78274386,2015-01-02,13.0,2015-01-02 13:00:00
3,5659,78274389,2015-01-02,19.0,2015-01-02 19:00:00
4,5659,78274390,2015-01-04,16.0,2015-01-04 16:00:00


In [20]:
df_off_type = pd.read_parquet(FOLDER + '/NIBRS_OFFENSE_TYPE.parquet')
df_off_type = df_off_type[['OFFENSE_CODE','OFFENSE_NAME','OFFENSE_CATEGORY_NAME']]

In [21]:
df_agencies = pd.read_parquet(FOLDER + '/AGENCIES.parquet')
df_agencies = df_agencies[['AGENCY_ID','UCR_AGENCY_NAME','NCIC_AGENCY_NAME','PUB_AGENCY_NAME','COUNTY_NAME']]
df_agencies = df_agencies.drop_duplicates(subset=['AGENCY_ID']).reset_index(drop=True)
df_agencies

,AGENCY_ID,UCR_AGENCY_NAME,NCIC_AGENCY_NAME,PUB_AGENCY_NAME,COUNTY_NAME
0,5659,ROCKFORD,ROCKFORD PD,Rockford,OGLE; WINNEBAGO
1,5655,WINNEBAGO,WINNEBAGO CO SO ROCKFORD,Winnebago,WINNEBAGO
2,5656,CHERRY VALLEY,CHERRY VALLEY PD,Cherry Valley,BOONE; WINNEBAGO
3,5657,LOVES PARK,LOVES PARK PD IL,Loves Park,BOONE; WINNEBAGO
4,5661,ROSCOE,ROSCOE PD,Roscoe,WINNEBAGO
...,...,...,...,...,...
700,44037,LOCKPORT TOWNSHIP PARK DISTRICT,None,Lockport Township Park District,WILL
701,44038,KASKASKIA COLLEGE,None,Kaskaskia College,CLINTON
702,44039,AURORA UNIVERSITY,None,Aurora University,KANE
703,44367,MOUNDS,None,Mounds,PULASKI


In [22]:
df_incident_agency = df_incident.merge(
    df_agencies,
    on='AGENCY_ID',
    how='left'
)
df_incident_agency

,AGENCY_ID,INCIDENT_ID,INCIDENT_DATE,INCIDENT_HOUR,INCIDENT_DATETIME,UCR_AGENCY_NAME,NCIC_AGENCY_NAME,PUB_AGENCY_NAME,COUNTY_NAME
0,5659,78274377,2015-01-02,0.0,2015-01-02 00:00:00,ROCKFORD,ROCKFORD PD,Rockford,OGLE; WINNEBAGO
1,5659,78274378,2015-01-02,21.0,2015-01-02 21:00:00,ROCKFORD,ROCKFORD PD,Rockford,OGLE; WINNEBAGO
2,5659,78274386,2015-01-02,13.0,2015-01-02 13:00:00,ROCKFORD,ROCKFORD PD,Rockford,OGLE; WINNEBAGO
3,5659,78274389,2015-01-02,19.0,2015-01-02 19:00:00,ROCKFORD,ROCKFORD PD,Rockford,OGLE; WINNEBAGO
4,5659,78274390,2015-01-04,16.0,2015-01-04 16:00:00,ROCKFORD,ROCKFORD PD,Rockford,OGLE; WINNEBAGO
...,...,...,...,...,...,...,...,...,...
1825417,5684,208138581,2024-12-11,12.0,2024-12-11 12:00:00,CHICAGO,CHICAGO PD,Chicago,COOK
1825418,5684,208138582,2024-10-31,18.0,2024-10-31 18:00:00,CHICAGO,CHICAGO PD,Chicago,COOK
1825419,5684,208138583,2024-03-11,0.0,2024-03-11 00:00:00,CHICAGO,CHICAGO PD,Chicago,COOK
1825420,5684,199835404,2024-09-04,23.0,2024-09-04 23:00:00,CHICAGO,CHICAGO PD,Chicago,COOK


In [23]:
df_offense_detail = df_offense.merge(
    df_off_type,
    on='OFFENSE_CODE',
    how='left'
)
df_offense_detail

,OFFENSE_ID,INCIDENT_ID,OFFENSE_CODE,OFFENSE_NAME,OFFENSE_CATEGORY_NAME
0,172305650,143439540,13B,Simple Assault,Assault Offenses
1,172305651,143439541,13B,Simple Assault,Assault Offenses
2,172305652,143439542,11A,Rape,Sex Offenses
3,172305654,143439544,13B,Simple Assault,Assault Offenses
4,172305655,143439545,23F,Theft From Motor Vehicle,Larceny/Theft Offenses
...,...,...,...,...,...
1937372,246599230,208138581,26A,False Pretenses/Swindle/Confidence Game,Fraud Offenses
1937373,246599231,208138582,290,Destruction/Damage/Vandalism of Property,Destruction/Damage/Vandalism of Property
1937374,246599232,208138583,26F,Identity Theft,Fraud Offenses
1937375,237085432,199835404,520,Weapon Law Violations,Weapon Law Violations


In [24]:
df_offense_final = df_offense_detail.merge(
    df_incident_agency,
    on='INCIDENT_ID',
    how='left'
)
df_offense_final

,OFFENSE_ID,INCIDENT_ID,OFFENSE_CODE,OFFENSE_NAME,OFFENSE_CATEGORY_NAME,AGENCY_ID,INCIDENT_DATE,INCIDENT_HOUR,INCIDENT_DATETIME,UCR_AGENCY_NAME,NCIC_AGENCY_NAME,PUB_AGENCY_NAME,COUNTY_NAME
0,172305650,143439540,13B,Simple Assault,Assault Offenses,4677,2021-01-24,1.0,2021-01-24 01:00:00,QUINCY,QUINCY PD,Quincy,ADAMS
1,172305651,143439541,13B,Simple Assault,Assault Offenses,4677,2021-02-26,15.0,2021-02-26 15:00:00,QUINCY,QUINCY PD,Quincy,ADAMS
2,172305652,143439542,11A,Rape,Sex Offenses,4677,2021-02-27,14.0,2021-02-27 14:00:00,QUINCY,QUINCY PD,Quincy,ADAMS
3,172305654,143439544,13B,Simple Assault,Assault Offenses,4677,2021-02-27,22.0,2021-02-27 22:00:00,QUINCY,QUINCY PD,Quincy,ADAMS
4,172305655,143439545,23F,Theft From Motor Vehicle,Larceny/Theft Offenses,4677,2021-02-28,2.0,2021-02-28 02:00:00,QUINCY,QUINCY PD,Quincy,ADAMS
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1937372,246599230,208138581,26A,False Pretenses/Swindle/Confidence Game,Fraud Offenses,5684,2024-12-11,12.0,2024-12-11 12:00:00,CHICAGO,CHICAGO PD,Chicago,COOK
1937373,246599231,208138582,290,Destruction/Damage/Vandalism of Property,Destruction/Damage/Vandalism of Property,5684,2024-10-31,18.0,2024-10-31 18:00:00,CHICAGO,CHICAGO PD,Chicago,COOK
1937374,246599232,208138583,26F,Identity Theft,Fraud Offenses,5684,2024-03-11,0.0,2024-03-11 00:00:00,CHICAGO,CHICAGO PD,Chicago,COOK
1937375,237085432,199835404,520,Weapon Law Violations,Weapon Law Violations,5684,2024-09-04,23.0,2024-09-04 23:00:00,CHICAGO,CHICAGO PD,Chicago,COOK


In [25]:
offense_category_map = {
    # ===================== THEFT =====================
    "Pocket-picking": "THEFT",
    "Burglary/Breaking & Entering": "THEFT",
    "Motor Vehicle Theft": "THEFT",
    "Shoplifting": "THEFT",
    "Theft From Building": "THEFT",
    "Theft From Coin-Operated Machine or Device": "THEFT",
    "Theft From Motor Vehicle": "THEFT",
    "Theft of Motor Vehicle Parts or Accessories": "THEFT",
    "All Other Larceny": "THEFT",
    "Stolen Property Offenses": "THEFT",

    # ===================== ROBBERY =====================
    "Robbery": "ROBBERY",
    "Purse-snatching": "ROBBERY",

    # ===================== ASSAULT =====================
    "Aggravated Assault": "ASSAULT",
    "Simple Assault": "ASSAULT",
    "Intimidation": "ASSAULT",
    "Murder and Nonnegligent Manslaughter": "ASSAULT",
    "Negligent Manslaughter": "ASSAULT",
    "Justifiable Homicide": "ASSAULT",
    "Rape": "ASSAULT",
    "Sodomy": "ASSAULT",
    "Sexual Assault With An Object": "ASSAULT",
    "Criminal Sexual Contact": "ASSAULT",
    "Incest": "ASSAULT",
    "Statutory Rape": "ASSAULT",
    "Kidnapping/Abduction": "ASSAULT",
    "Human Trafficking, Commercial Sex Acts": "ASSAULT",
    "Human Trafficking, Involuntary Servitude": "ASSAULT",
    "Animal Cruelty": "ASSAULT",
    "Peeping Tom": "ASSAULT",

    # ===================== OTHER =====================
    "Arson": "OTHER",
    "Extortion/Blackmail": "OTHER",
    "Not Specified": "OTHER",
    "Pornography/Obscene Material": "OTHER",
    "Prostitution": "OTHER",
    "Assisting or Promoting Prostitution": "OTHER",
    "Purchasing Prostitution": "OTHER",
    "Weapon Law Violations": "OTHER",
    "Counterfeiting/Forgery": "OTHER",
    "False Pretenses/Swindle/Confidence Game": "OTHER",
    "Flight to Avoid Deportation": "OTHER",
    "Flight to Avoid Prosecution": "OTHER",
    "Harboring Escapee/Concealing from Arrest": "OTHER",
    "Failure to Register as a Sex Offender": "OTHER",
    "Treason": "OTHER",
    "Espionage": "OTHER",
    "Bad Checks": "OTHER",
    "Illegal Entry into the United States": "OTHER",
    "False Citizenship": "OTHER",
    "Smuggling Aliens": "OTHER",
    "Re-entry after Deportation": "OTHER",
    "Curfew/Loitering/Vagrancy Violations": "OTHER",
    "Disorderly Conduct": "OTHER",
    "Credit Card/Automated Teller Machine Fraud": "OTHER",
    "Impersonation": "OTHER",
    "Welfare Fraud": "OTHER",
    "Wire Fraud": "OTHER",
    "Identity Theft": "OTHER",
    "Hacking/Computer Invasion": "OTHER",
    "Embezzlement": "OTHER",
    "Destruction/Damage/Vandalism of Property": "OTHER",
    "Drug/Narcotic Violations": "OTHER",
    "Drug Equipment Violations": "OTHER",
    "Betting/Wagering": "OTHER",
    "Operating/Promoting/Assisting Gambling": "OTHER",
    "Gambling Equipment Violation": "OTHER",
    "Sports Tampering": "OTHER",
    "Bribery": "OTHER",
    "Export Violations": "OTHER",
    "Driving Under the Influence": "OTHER",
    "Drunkenness": "OTHER",
    "Family Offenses, Nonviolent": "OTHER",
    "Liquor Law Violations": "OTHER",
    "Trespass of Real Property": "OTHER",
    "All Other Offenses": "OTHER",
    "Failure to Appear": "OTHER",
    "Federal Resource Violations": "OTHER",
    "Perjury": "OTHER",
    "Runaway": "OTHER",
    "Import Violations": "OTHER",
    "Federal Liquor Offenses": "OTHER",
    "Federal Tobacco Offenses": "OTHER",
    "Money Laundering": "OTHER",
    "Wildlife Trafficking": "OTHER",
    "Explosives Violation": "OTHER",
    "Weapons of Mass Destruction": "OTHER",
    "Violation of National Firearm Act of 1934": "OTHER",
}

In [26]:
df_offense_final["CRIME_GROUP"] = df_offense_final["OFFENSE_NAME"].map(offense_category_map)
df_offense_violent = df_offense_final.loc[(df_offense_final.CRIME_GROUP != "OTHER") ]

In [27]:
df_offense_violent

,OFFENSE_ID,INCIDENT_ID,OFFENSE_CODE,OFFENSE_NAME,OFFENSE_CATEGORY_NAME,AGENCY_ID,INCIDENT_DATE,INCIDENT_HOUR,INCIDENT_DATETIME,UCR_AGENCY_NAME,NCIC_AGENCY_NAME,PUB_AGENCY_NAME,COUNTY_NAME,CRIME_GROUP
0,172305650,143439540,13B,Simple Assault,Assault Offenses,4677,2021-01-24,1.0,2021-01-24 01:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
1,172305651,143439541,13B,Simple Assault,Assault Offenses,4677,2021-02-26,15.0,2021-02-26 15:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
2,172305652,143439542,11A,Rape,Sex Offenses,4677,2021-02-27,14.0,2021-02-27 14:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
3,172305654,143439544,13B,Simple Assault,Assault Offenses,4677,2021-02-27,22.0,2021-02-27 22:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,ASSAULT
4,172305655,143439545,23F,Theft From Motor Vehicle,Larceny/Theft Offenses,4677,2021-02-28,2.0,2021-02-28 02:00:00,QUINCY,QUINCY PD,Quincy,ADAMS,THEFT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1937363,246599225,208138576,23C,Shoplifting,Larceny/Theft Offenses,5684,2024-09-30,12.0,2024-09-30 12:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT
1937364,246599226,208138577,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-11-20,13.0,2024-11-20 13:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT
1937366,246599227,208138578,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-03-01,9.0,2024-03-01 09:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT
1937367,246599228,208138579,23H,All Other Larceny,Larceny/Theft Offenses,5684,2024-03-08,10.0,2024-03-08 10:00:00,CHICAGO,CHICAGO PD,Chicago,COOK,THEFT


In [28]:
# Count of missing values per column
missing_summary = df_offense_violent.isna().sum()
print(missing_summary)

OFFENSE_ID                   0
INCIDENT_ID                  0
OFFENSE_CODE                 0
OFFENSE_NAME                 0
OFFENSE_CATEGORY_NAME        0
AGENCY_ID                    0
INCIDENT_DATE                0
INCIDENT_HOUR            14481
INCIDENT_DATETIME        14481
UCR_AGENCY_NAME              0
NCIC_AGENCY_NAME           186
PUB_AGENCY_NAME              0
COUNTY_NAME                  0
CRIME_GROUP                  0
dtype: int64


In [29]:
df_offense_violent_clean = df_offense_violent.dropna()
print(f"Before dropping NaN values: {len(df_offense_violent)}")
print(f"After dropping NaN values: {len(df_offense_violent_clean)}")
print(f"Data dropped = {round((len(df_offense_violent)-len(df_offense_violent_clean))/len(df_offense_violent)*100,2)}")

Before dropping NaN values: 1287231
After dropping NaN values: 1272576
Data dropped = 1.14


In [30]:
len(df_offense_violent_clean.INCIDENT_ID)

1272576

In [31]:
df_drop = df_offense_violent_clean.drop_duplicates(subset='INCIDENT_ID')

In [32]:
df_offense_violent_clean.to_parquet('IllinoisViolentCrime.parquet')